# hitl_test.ipynb — 执行中 HITL 全链路验证 (6用例)

按 spec §8 验收表覆盖: ⑧ mid_clarify 检索反馈追问(节点级) ⑨ replan_check 出口路由优先级 ⑩ degrade 三分支 ⑪ degrade 一次性防循环 ⑫ budget 补充/收尾 ⑬ pdf_confirm 回归 + error_streak 计数。节点与路由行为真实, 仅 LLM 调用与 interrupt 以测试替身接管(与 tests/test_smoke.py 同体系)。

## Cell 1 环境与测试替身 (setup, 自包含 — 与 clarify_test.ipynb Cell 1 逐字一致)

In [ ]:
import sys, asyncio, json
from pathlib import Path
from unittest.mock import patch

ROOT = Path.cwd().parent if Path.cwd().name == "lawApp_LangGraph" else Path.cwd()
sys.path.insert(0, str(ROOT))

# LLM 替身(移植 tests/test_smoke.py 体系)
from langchain_core.runnables import Runnable

class _FakeMsg:
    def __init__(self, content="", tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls or []

class _FakeVerdict:
    def __init__(self, **kw):
        self.plan = list(kw.get("plan", ()))
        self.reasoning = list(kw.get("reasoning", ()))
        self.need_clarification = kw.get("need_clarification", False)
        self.question = kw.get("question", "")
        self.high_risk = kw.get("high_risk", False)
        self.needs_replan = kw.get("needs_replan", False)
        self.reason = kw.get("reason", "")
        self.applicable = kw.get("applicable", True)
        self.element_updates = list(kw.get("element_updates", ()))
        self.na_keys = list(kw.get("na_keys", ()))
        self.promote_keys = list(kw.get("promote_keys", ()))
        self.questions = list(kw.get("questions", ()))
        self.done = kw.get("done", False)
        self.insufficient_reason = kw.get("insufficient_reason", "none")

class _FakeChain(Runnable):
    def __init__(self, result=None):
        self.result = result
    def invoke(self, _inp, config=None, **kwargs):
        return self.result
    async def ainvoke(self, _inp, config=None, **kwargs):
        return self.result
    async def astream(self, _inp, config=None, **kwargs):
        yield _FakeMsg("测试回答")

class _FakeLLM(Runnable):
    def __init__(self, state):
        self.state = state
    def invoke(self, msgs, config=None, **kwargs):
        return self.state["executor_result"]
    def with_structured_output(self, schema):
        return _FakeChain(result=self.state["plan_result"])
    def bind_tools(self, tools):
        return self
    async def ainvoke(self, msgs, config=None, **kwargs):
        return self.state["executor_result"]
    async def astream(self, _prompt, config=None, **kwargs):
        yield _FakeMsg("测试回答")

import lawApp_LangGraph.LangGraph_lawApp as app
import lawApp_LangGraph.tools.rag_tools as rag_tools

class _ToolLLM:
    async def astream(self, _prompt):
        yield _FakeMsg("分析结果:测试回答")

_ctrl = {"plan_result": _FakeVerdict(), "executor_result": None}
_p_planner = patch.object(app, "get_planner_llm", lambda: _FakeLLM(_ctrl))
_p_executor = patch.object(app, "get_executor_llm", lambda: _FakeLLM(_ctrl))
_p_rag = patch.object(rag_tools, "_get_llm", lambda: _ToolLLM())
_p_planner.start(); _p_executor.start(); _p_rag.start()
print("setup ok")

## 用例⑧ mid_clarify 检索反馈追问 (节点级: resume 增强 / 空答放行)

In [ ]:
from lawApp_LangGraph.state import AgentState, RetrievedDocument

async def case8():
    # mid_clarify 消费 executor LLM 的 structured output:
    # 用真 MidClarifySchema 实例喂同一替身(与节点读取的 v.question/v.element_key 同形)
    _ctrl["plan_result"] = app.MidClarifySchema(
        question="你的房子是婚前买的还是婚后买的?",
        element_key="property",
    )

    st = AgentState(
        query="离婚房子怎么分",
        rag_documents=[
            RetrievedDocument(
                case_number="(2023)京01民终1号",
                chunk_text="婚后共同还贷情形下,房产原则上均等分割…",
                year="2023",
            )
        ],
        mid_clarify_used=False,
    )

    # resume 非空 → query 织入「[检索反馈追问]/[用户澄清]」,
    # case_elements 深拷贝后按 element_key 记录(来源 mid_clarify)
    with patch.object(app, "interrupt", lambda p: "婚前我付首付,婚后共同还贷"):
        upd = await app.mid_clarify_node(st)
    assert "[检索反馈追问]" in upd["query"]
    assert "[用户澄清] 婚前我付首付,婚后共同还贷" in upd["query"]
    assert upd["mid_clarify_used"] is True
    assert upd["case_elements"].elements[2].status == "known"       # property
    assert upd["case_elements"].elements[2].updated_by == "mid_clarify"
    assert upd["hitl_event"]["type"] == "mid_clarify"

    # resume 空回答 → 静默放行,由 replanner 联网兜底(只置一次性标记)
    with patch.object(app, "interrupt", lambda p: ""):
        upd = await app.mid_clarify_node(st)
    assert upd == {"mid_clarify_used": True}
    print("⑧ mid_clarify 节点级 OK(resume增强/空答放行)")

await case8()


## 用例⑨ replan_check 出口路由优先级 (finalize / budget / mid_clarify / replanner)

In [ ]:
from lawApp_LangGraph.state import AgentState, ToolCallRecord

def case9():
    # 优先级 1: 质量通过(不需重规划) → finalize
    assert app.route_after_replan_check(AgentState(query="x")) == "finalize"

    # 优先级 4: not_found(案例库覆盖不到) → replanner, 不问用户
    assert app.route_after_replan_check(
        AgentState(query="x", replan_needed=True, insufficient_reason="not_found")
    ) == "replanner"

    # 优先级 3: vague(问题笼统)且未用过 → mid_clarify(先问人)
    assert app.route_after_replan_check(
        AgentState(query="x", replan_needed=True, insufficient_reason="vague")
    ) == "mid_clarify"
    # vague 但已用过 → replanner(一次性防循环)
    assert app.route_after_replan_check(
        AgentState(query="x", replan_needed=True, insufficient_reason="vague",
                   mid_clarify_used=True)
    ) == "replanner"

    # 优先级 2: 预算耗尽(工具调用数达 MAX_ROUNDS=10) → 未问过 hitl_budget
    st_budget = AgentState(
        query="x", replan_needed=True,
        tool_calls=[ToolCallRecord(step_id=i, tool_name="t")
                    for i in range(app.MAX_ROUNDS)],
    )
    assert app.route_after_replan_check(st_budget) == "hitl_budget"
    # 耗尽且已问过 → finalize(带现有材料收口)
    st_budget2 = st_budget.model_copy(update={"budget_hitl_used": True})
    assert app.route_after_replan_check(st_budget2) == "finalize"

    # 优先级序: 预算耗尽与 vague 并存时 budget 分支先短路(不进 mid_clarify)
    st_both = st_budget.model_copy(update={"insufficient_reason": "vague"})
    assert app.route_after_replan_check(st_both) == "hitl_budget"
    print("⑨ 路由优先级 OK")

case9()


## 用例⑩⑪ degrade 三分支 + 一次性防循环

In [ ]:
from lawApp_LangGraph.state import AgentState, PlanStep

def case10():
    # hitl_degrade_node 为同步节点, 直接调用(brief 原文的 await 按实际签名修正)
    st = AgentState(
        query="x",
        plan=[PlanStep(step_id=1, description="检索",
                       tool_name="retrieve_legal_knowledge")],
        current_step_index=0, error_streak=2, degrade_used=False,
    )

    # retry → 计数清零 + replan_needed, 路由回 replanner 重试
    with patch.object(app, "interrupt", lambda p: "重试"):
        upd = app.hitl_degrade_node(st)
    assert upd["degrade_used"] is True and upd["error_streak"] == 0
    assert upd["replan_needed"] is True and "重试" in upd["replan_reason"]
    assert app.route_after_degrade(
        AgentState(query="x", replan_needed=True)) == "replanner"

    # abort → 热线中止文案 + finalize 出口
    with patch.object(app, "interrupt", lambda p: "终止"):
        upd = app.hitl_degrade_node(st)
    assert "中止" in upd["final_answer"]
    assert app.route_after_degrade(
        AgentState(query="x", final_answer=upd["final_answer"])) == "finalize"

    # skip(默认) → 只清计数, replan_check 质量门控收口
    with patch.object(app, "interrupt", lambda p: "跳过"):
        upd = app.hitl_degrade_node(st)
    assert upd["error_streak"] == 0 and "final_answer" not in upd
    assert upd["hitl_event"]["choice"] == "skip"
    assert app.route_after_degrade(AgentState(query="x")) == "replan_check"

    # ⑪ 一次性: degrade_used=True 后 executor/merge 两处降级门均不再触发
    st_used = AgentState(
        query="x", error_streak=5, degrade_used=True,
        plan=[PlanStep(step_id=1, description="检索",
                       tool_name="retrieve_legal_knowledge")],
        current_step_index=0,
    )
    assert app.route_after_merge(st_used) != "hitl_degrade"
    assert app.route_after_executor(st_used) != "hitl_degrade"
    # 对照组: 未用过且连续失败达阈值 → 触发降级询问
    assert app.route_after_merge(
        AgentState(query="x", error_streak=2)) == "hitl_degrade"
    print("⑩⑪ degrade 分支与一次性 OK")

case10()


## 用例⑫ budget 补充 / 收尾两分支

In [ ]:
from lawApp_LangGraph.state import AgentState, EvaluationResult

def case12():
    # hitl_budget_node 为同步节点, 直接调用(brief 原文的 await 按实际签名修正)
    st = AgentState(
        query="x",
        evaluation=EvaluationResult(
            total=5, correct_count=1, ambiguous_count=1, incorrect_count=3,
            quality_verdict="不足,建议进行网络搜索补充",
        ),
    )

    # 补充分支(非空文本且不含收尾关键词) → query 织入 + 最后一次 replan
    with patch.object(app, "interrupt", lambda p: "对方偷偷转移了财产"):
        upd = app.hitl_budget_node(st)
    assert upd["budget_hitl_used"] is True
    assert "[用户补充信息] 对方偷偷转移了财产" in upd["query"]
    assert upd["replan_needed"] is True
    assert upd["hitl_event"]["choice"] == "supplement"
    assert app.route_after_budget(
        AgentState(query="x", replan_needed=True)) == "replanner"

    # 收尾分支(关键词「收尾」) → 不写 final_answer/replan, 直接 finalize
    with patch.object(app, "interrupt", lambda p: "收尾"):
        upd = app.hitl_budget_node(st)
    assert upd["budget_hitl_used"] is True
    assert "final_answer" not in upd and not upd.get("replan_needed")
    assert upd["hitl_event"]["choice"] == "finish"
    assert app.route_after_budget(AgentState(query="x")) == "finalize"

    # 收尾分支(空回答) → 节点层空值同样收尾
    with patch.object(app, "interrupt", lambda p: ""):
        upd = app.hitl_budget_node(st)
    assert upd["hitl_event"]["choice"] == "finish"
    print("⑫ budget 补充/收尾 OK")

case12()


## 用例⑬ pdf_confirm 存量回归 + error_streak 计数 (含收尾 teardown)

In [ ]:
from lawApp_LangGraph.state import AgentState, PlanStep

async def case13():
    # pdf_confirm 存量行为回归: 计划含 markdown_to_pdf 且未确认 → interrupt,
    # 用户拒绝 → 步骤标记 done(跳过) + pdf_confirmed=True(不再询问)
    st = AgentState(
        query="出份报告",
        plan=[PlanStep(step_id=1, description="生成PDF",
                       tool_name="markdown_to_pdf")],
        current_step_index=0, pdf_confirmed=False,
    )
    with patch.object(app, "interrupt", lambda p: False):  # 用户拒绝
        upd = await app.executor_node(st)
    assert upd["plan"][0].status == "done"       # 跳过标记
    assert upd["current_step_index"] == 1
    assert upd["pdf_confirmed"] is True
    assert upd["hitl_event"]["type"] == "pdf_confirm"
    assert upd["hitl_event"]["confirmed"] is False

    # error_streak 计数: executor 两次参数提取失败(ainvoke 返回 None,
    # 非 AIMessage → RuntimeError → 重试仍 None) → 步骤 failed + 计数 +1
    _ctrl["executor_result"] = None
    st2 = AgentState(
        query="x",
        plan=[PlanStep(step_id=1, description="检索",
                       tool_name="retrieve_legal_knowledge")],
        current_step_index=0, error_streak=0,
    )
    upd2 = await app.executor_node(st2)
    assert upd2["error_streak"] == 1
    assert upd2["plan"][0].status == "failed"
    assert "LLM 参数提取失败" in upd2["error"]
    print("⑬ pdf_confirm回归 + error_streak OK")

await case13()
_p_planner.stop(); _p_executor.stop(); _p_rag.stop()
print("ALL PASSED")
